# Image

In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
model_path = "./pose_landmarker.task"

In [4]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE)

mp_image = mp.Image.create_from_file('./image.jpg')

with PoseLandmarker.create_from_options(options) as landmarker:
  # The landmarker is initialized. Use it here.
  # ...
  pose_landmarker_result = landmarker.detect(mp_image)
    

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1769700365.886620   16772 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769700366.040360   16772 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769700366.189437   16772 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [ ]:
from analysis import has_display, visualize_qb_manual
import cv2

# Run it
result_img = visualize_qb_manual('image.jpg', pose_landmarker_result)

cv2.imwrite('output_image.jpg', result_img)
print("Saved annotated image to output_image.jpg")

if has_display():
    cv2.imshow('Manual QB Trace', result_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("No display detected (headless environment) - skipping cv2.imshow.")


# Video

In [ ]:
from analysis import analyze_video, render_annotated_video, summarize_releases

# Wrist speed, release time(s), and hip-shoulder separation - see analysis.py.
# pose_world_landmarks (metric-scale 3D) are used instead of normalized image
# coordinates so the numbers aren't distorted by camera perspective.
analysis_back = analyze_video('qb-throw.mp4', model_path=model_path, view='back')

print(f"Detected {len(analysis_back.release_frames)} candidate release event(s):\n")
for event in summarize_releases(analysis_back):
    print(f"  t={event['time_s']:5.2f}s  release speed={event['speed_mps']:4.2f} m/s "
          f"({event['speed_mph']:4.1f} mph)  peak hip-shoulder separation="
          f"{event['peak_separation_deg']:5.1f} deg ({event['separation_lead_ms']:.0f}ms before release)")
print("\n(release speed is throwing-hand speed, a proxy for ball speed, not the ball itself)")

# Same overlay (skeleton + elbow angle), plus the metrics above and a RELEASE
# flash burned directly into the video instead of only printed to the console.
frame_count = render_annotated_video('qb-throw.mp4', 'qb-throw-annotated.mp4', analysis_back, model_path=model_path)
print(f"\nProcessed {frame_count} frames. Saved annotated video to qb-throw-annotated.mp4")


# More Angles & Consistency Across Reps

The same landmarks already extracted in `analyze_video()` also give elbow angle (at release and at "max cocking" - approximated as the pause in wrist speed right before the forward swing), trunk lean, and stride length (peak foot separation). Since a clip usually has several throwing reps, `summarize_consistency()` reports the mean/stddev/CV% of each metric across reps - useful for seeing how *repeatable* the mechanics are, not just any single throw's numbers.

In [ ]:
for event in summarize_releases(analysis_back):
    print(
        f"t={event['time_s']:5.2f}s  "
        f"elbow @release={event['elbow_angle_release_deg']:5.1f} deg  "
        f"elbow @max cocking={event['elbow_angle_max_cocking_deg']:5.1f} deg  "
        f"trunk lean={event['trunk_lean_release_deg']:4.1f} deg  "
        f"stride={event['stride_length_m']:.2f} m"
    )

from analysis import summarize_consistency

consistency = summarize_consistency(summarize_releases(analysis_back))
print("\nConsistency across reps (mean +/- std, CV%):")
if consistency is None:
    print("  Only one release event detected - nothing to compare.")
else:
    for key, stat in consistency.items():
        cv = f"{stat['cv_pct']:.0f}%" if stat['cv_pct'] is not None else "n/a"
        print(f"  {key:26s} {stat['mean']:7.2f} +/- {stat['std']:5.2f}   (CV {cv})")


# Ball Tracking (Experimental)

Everything above measures the throwing *hand*, not the ball - wrist speed is a proxy for release speed. `track_ball_release()` tries to track the actual football for a short window after each release instead, using a lightweight classical-CV detector (frame differencing in a small region ahead of the hand) self-calibrated to real-world units from the thrower's own body (meters-per-pixel derived from a known body segment's world-space length vs. its pixel length in the same frame).

This is **not** a trained object detector, and it only reports a release when it finds a consistent, physically plausible track - it's expected to come back empty on a busy background, or on a back-view camera angle where the ball moves mostly in depth rather than laterally across the frame (a side view gives it a much easier time).

In [ ]:
from analysis import track_ball_release

ball_releases = track_ball_release('qb-throw.mp4', analysis_back, model_path=model_path)
if not ball_releases:
    print("No confident ball track found in this clip (expected on a busy back-view angle - try a side view).")
else:
    for ball in ball_releases:
        print(f"frame={ball.frame_idx}  ball speed={ball.speed_mps:.2f} m/s ({ball.speed_mph:.1f} mph)  "
              f"launch angle={ball.angle_deg:.1f} deg  ({len(ball.positions)} detections)")


# Metrics Chart

In [ ]:
import matplotlib.pyplot as plt
from analysis import compare_views

fig = compare_views({'back': analysis_back})
fig.savefig('throw-metrics.png', dpi=120)
plt.show()


# Multi-View Comparison (Side & Front)

`analyze_video()` and `compare_views()` work on a video from any camera angle - MediaPipe assigns landmarks by the subject's own anatomical left/right, not by where the camera is standing, so no code changes are needed for a side or front clip.

This repo currently only ships a back-view clip (`qb-throw.mp4`). Drop a `qb-throw-side.mp4` and/or `qb-throw-front.mp4` of the *same* throwing session next to it and re-run the cell below to compare - each view is plotted with its own detected release aligned to t=0.

This is **not** true synchronized 3D triangulation (that needs calibrated, timestamp-aligned cameras filming the same instant), but it does let you sanity-check whether independent single-view measurements roughly agree - useful given a 2D-plane joint angle or speed estimate can be distorted by which way the camera happens to be facing.

In [ ]:
from pathlib import Path

view_paths = {
    'side': 'qb-throw-side.mp4',
    'front': 'qb-throw-front.mp4',
}

results = {'back': analysis_back}  # already computed above
for view_name, path in view_paths.items():
    if not Path(path).exists():
        print(f"Skipping '{view_name}': {path} not found. Add this file to compare that angle.")
        continue
    results[view_name] = analyze_video(path, model_path=model_path, view=view_name)

if len(results) > 1:
    fig = compare_views(results)
    fig.savefig('throw-metrics-multiview.png', dpi=120)
    plt.show()
else:
    print("Only the back view is available - add qb-throw-side.mp4 / qb-throw-front.mp4 to compare angles.")
